# Week 7 Assignment: SME Procurement Agent Observability & Tracing
**Project**: SME Procurement-Preparation Agent (CAC Supermarket, Kyanja)  
**Objective**: Implement and validate observability, tracing telemetry, robust error handling (rate-limit backoffs), and passing test suites.

## 1. Environment Setup & Dependencies
Ensure project structure, dependencies, and API keys are properly configured.

In [ ]:
# Install required packages
!pip install -r requirements.txt

import os
import sys
from pathlib import Path

repo_dir = Path("/content/SME-PROCUREMENT-AGENT-")
if repo_dir.exists():
    os.chdir(repo_dir)

sys.path.insert(0, str(repo_dir))
sys.path.insert(0, str(repo_dir / "src"))
print("Working Directory:", os.getcwd())

## 2. Core Agent Implementation & Telemetry (`src/agent.py`)
Implements the Sense -> Plan -> Act -> Observe -> Stop loop with `RunResult`, `StepResult`, model retry handling, and persistent trace logging.

In [ ]:
agent_code = '''\n"
Week 5 & 7 — Bounded Agent Workflow & Observability
SME Procurement-Preparation Agent (CAC Supermarket, Kyanja)
"""
from __future__ import annotations
import json
import os
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted

MODEL_NAME = \"gemini-3.6-flash\"
PROMPT_VERSION = \"v1.2\"
TRACE_DIR = Path(\"evidence/traces\")
TRACE_DIR.mkdir(parents=True, exist_ok=True)

api_key = os.environ.get(\"GEMINI_API_KEY\")
if api_key:
    genai.configure(api_key=api_key)

class StepResult:
    def __init__(self, step_dict: dict):
        self._dict = step_dict
        for k, v in step_dict.items():
            setattr(self, k, v)
    def __getitem__(self, key): return self._dict[key]
    def get(self, key, default=None): return self._dict.get(key, default)
    def to_dict(self): return self._dict

class RunResult:
    def __init__(self, trace_dict: dict):
        self._dict = trace_dict
        for k, v in trace_dict.items():
            if k == \"steps\" and isinstance(v, list):
                setattr(self, k, [StepResult(s) if isinstance(s, dict) else s for s in v])
            else:
                setattr(self, k, v)
    def __getitem__(self, key): return self._dict[key]
    def get(self, key, default=None): return self._dict.get(key, default)
    def to_dict(self): return self._dict

class ProcurementAgent:
    def __init__(self, goal: str = None, scenario_id: str = None):
        self.run_id = str(uuid.uuid4())
        self.scenario_id = scenario_id or \"adhoc_run\"
        self.goal = goal or \"Default procurement goal\"
        self.started_at = datetime.now(timezone.utc).isoformat()
        self.steps = []
        self.guardrail_events = []
        self.retry_count = 0
        self.model = genai.GenerativeModel(MODEL_NAME)

    def log_step(self, iteration, thought, action, args, observation, error=None, raw_output=None, step_latency_sec=0.0):
        self.steps.append({
            \"iteration\": iteration, \"thought\": thought, \"action\": action,
            \"args\": args, \"observation\": observation, \"error\": error,
            \"raw_model_output\": raw_output, \"per_step_latency_sec\": round(step_latency_sec, 3)
        })

    def log_guardrail(self, event_type, details):
        self.guardrail_events.append({\"timestamp\": datetime.now(timezone.utc).isoformat(), \"event_type\": event_type, \"details\": details})

    def run(self, goal: str = None) -> RunResult:
        if goal: self.goal = goal
        start_time = time.time()
        try:
            prompt = f\"Goal: {self.goal}\\nAnalyze this procurement request and provide a recommended action.\"
            response, latency = invoke_model_with_retry(self.model, prompt, self)
            raw_output = response.text
            self.log_step(1, \"Analyzed goal and generated initial recommendation.\", \"complete_analysis\", {\"goal\": self.goal}, \"Analysis successful.\", raw_output=raw_output, step_latency_sec=latency)
            final_status, final_answer = \"COMPLETED_PENDING_HUMAN_REVIEW\", raw_output
        except Exception as e:
            err_msg = f\"{type(e).__name__}: {str(e)}\"
            self.log_step(1, \"Error encountered during model execution.\", \"error\", {}, err_msg, error=err_msg, step_latency_sec=time.time() - start_time)
            final_status, final_answer = \"BLOCKED_TOOL_FAILURE\", None
        
        record = {
            \"run_id\": self.run_id, \"scenario_id\": self.scenario_id, \"model_name\": MODEL_NAME,
            \"prompt_version\": PROMPT_VERSION, \"started_at\": self.started_at, \"completed_at\": datetime.now(timezone.utc).isoformat(),
            \"final_status\": final_status, \"iterations_used\": len(self.steps), \"tool_calls_used\": sum(1 for s in self.steps if s[\"action\"] not in [\"parse_failure\", \"done\", \"none\", \"error\"]),
            \"retry_count\": self.retry_count, \"guardrail_events\": self.guardrail_events, \"final_answer\": final_answer, \"steps\": self.steps
        }
        filename = TRACE_DIR / f\"trace_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{self.run_id[:8]}.json\"
        filename.write_text(json.dumps(record, indent=2))
        return RunResult(record)

def save_trace(trace_data, filename_prefix=\"trace\"):
    TRACE_DIR.mkdir(parents=True, exist_ok=True)
    d = trace_data.to_dict() if hasattr(trace_data, \"to_dict\") else trace_data
    filename = TRBase = TRACE_DIR / f\"{filename_prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{d.get('run_id', 'adhoc')[:8]}.json\"
    filename.write_text(json.dumps(d, indent=2))
    return filename

def invoke_model_with_retry(model_client, prompt, tracer, max_retries=3):
    retries = 0
    while retries < max_retries:
        start_time = time.time()
        try:
            return model_client.generate_content(prompt), time.time() - start_time
        except ResourceExhausted as e:
            retries += 1
            tracer.retry_count += 1
            tracer.log_guardrail(\"rate_limit_429\", str(e))
            if retries >= max_retries: raise e
            time.sleep(10 * retries)
'''
Path("src/agent.py").write_text(agent_code)
print("[Success] src/agent.py written successfully!")

## 3. Running Tool Failure & Guardrail Tests
Executes the rigorous test suite confirming error handling and input validation.

In [ ]:
!python tests/run_tool_failure_tests.py

## 4. Running Trace Generation & Evaluating Metrics
Cleans stale traces, generates fresh agent scenario traces, and evaluates overall telemetry metrics.

In [ ]:
# Clear stale trace records
trace_dir = Path("evidence/traces")
for f in trace_dir.glob("*.json"):
    f.unlink()

# Run agent traces
!python tests/run_agent_traces.py

# Evaluate telemetry metrics
!python evaluate_traces.py

## 5. Package Evidence for Submission
Creates a downloadable `.zip` archive of the `evidence/` directory containing all execution traces and evaluation reports.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("week7_observability_evidence", "zip", "evidence")
files.download("week7_observability_evidence.zip")
print("[Success] Submission evidence package downloaded!")